In [1]:
import sympy as sp
import numpy as np
from sympy.physics.quantum.cg import CG
from itertools import product as iproduct
import time
import re

# =====================================================================
# 1. Exact T_op construction
# =====================================================================
def get_basis_info(i):
    if i < 2: return 0, i, 0
    elif i < 11: return 1, (i - 2) // 3, (i - 2) % 3
    else: return 2, (i - 11) // 5, (i - 11) % 5

def build_T_op_exact():
    U_schur = np.zeros((16, 16), dtype=object)
    paths_by_J = {
        0: [(0, sp.Rational(1,2)), (1, sp.Rational(1,2))],
        1: [(0, sp.Rational(1,2)), (1, sp.Rational(1,2)), (1, sp.Rational(3,2))],
        2: [(1, sp.Rational(3,2))]
    }
    col = 0
    for J in [0, 1, 2]:
        M_vals = [sp.Rational(M_2, 2) for M_2 in range(-int(2*J), int(2*J)+2, 2)]
        for s12, s123 in paths_by_J[J]:  
            for M in M_vals:             
                for i in range(16):
                    b = format(i, '04b')
                    m = [sp.Rational(1,2) if char == '0' else sp.Rational(-1,2) for char in b]
                    m12, m123 = m[0]+m[1], m[0]+m[1]+m[2]
                    if m123 + m[3] == M:
                        val = (CG(sp.Rational(1,2), m[0], sp.Rational(1,2), m[1], s12, m12).doit() * CG(s12, m12, sp.Rational(1,2), m[2], s123, m123).doit() * CG(s123, m123, sp.Rational(1,2), m[3], J, M).doit())
                        if val != 0: U_schur[i, col] = val
                col += 1
    U_full = np.kron(U_schur, U_schur)
    sy = np.array([[0, -sp.I], [sp.I, 0]], dtype=object)
    I2 = np.array([[1, 0], [0, 1]], dtype=object)
    S_list = [I2, sy, sy, sy, sy, sy, sy, I2]
    S_full = S_list[0]
    for i in range(1, 8): S_full = np.kron(S_full, S_list[i])
    P_vec = np.zeros((256, 256), dtype=object)
    pv = [0, 1, 3, 5, 2, 4, 6, 7]
    for r_in in range(256):
        b_in = format(r_in, '08b')
        b_out = ''.join([b_in[pv[k]] for k in range(8)])
        P_vec[int(b_out, 2), r_in] = 1
    sort_indices = list(range(256))
    sort_indices.sort(key=lambda x: (get_basis_info(x // 16)[0], get_basis_info(x % 16)[0], 
                                     get_basis_info(x // 16)[2], get_basis_info(x % 16)[2], 
                                     get_basis_info(x // 16)[1], get_basis_info(x % 16)[1]))
    P_sort = np.zeros((256, 256), dtype=object)
    for i_sorted, i_orig in enumerate(sort_indices): P_sort[i_sorted, i_orig] = 1
    T_op = S_full.dot(P_vec.T).dot(U_full).dot(P_sort.T)
    return T_op, np.conjugate(T_op).T

# =====================================================================
# 2. Hermitian matrix generation
# =====================================================================
real_symbols_set = set()

def create_herm_matrix(name, dim):
    mat = np.zeros((dim, dim), dtype=object)
    for i in range(dim):
        for j in range(i, dim):
            if i == j:
                sym = sp.Symbol(f'{name}_{i}{i}', real=True)
                mat[i, j] = sym
                real_symbols_set.add(sym)
            else:
                R = sp.Symbol(f'{name}R_{i}{j}', real=True)
                I = sp.Symbol(f'{name}I_{i}{j}', real=True)
                mat[i, j] = R + sp.I * I
                mat[j, i] = R - sp.I * I
                real_symbols_set.update([R, I])
    return mat

# =====================================================================
# 3. Partial trace & S3 symmetry utilities
# =====================================================================
DIMS = [2] * 8

def _flat(multi, dims):
    idx = 0
    for i, d in zip(multi, dims): idx = idx * d + i
    return idx

def partial_trace_np(C_mat, dims, trace_over):
    n = len(dims)
    keep = [i for i in range(n) if i not in trace_over]
    dk = [dims[i] for i in keep]
    dt = [dims[i] for i in trace_over]
    Dk = int(np.prod(dk)) if dk else 1
    res = np.zeros((Dk, Dk), dtype=object)
    kb = list(iproduct(*[range(d) for d in dk]))
    tb = list(iproduct(*[range(d) for d in dt]))
    for ri, ik in enumerate(kb):
        for ci, jk in enumerate(kb):
            val = 0
            for at in tb:
                row = [None]*n; col = [None]*n
                for p, k in enumerate(keep): row[k] = ik[p]; col[k] = jk[p]
                for p, k in enumerate(trace_over): row[k] = at[p]; col[k] = at[p]
                val += C_mat[_flat(row, dims), _flat(col, dims)]
            res[ri, ci] = val
    return res

def replace_subsystems_np(C_mat, dims, replace_idx):
    n = len(dims)
    D = int(np.prod(dims))
    keep = [i for i in range(n) if i not in replace_idx]
    dk = [dims[i] for i in keep]
    dr = [dims[i] for i in replace_idx]
    Dr = int(np.prod(dr)) if dr else 1
    C_red = partial_trace_np(C_mat, dims, replace_idx)
    result = np.zeros((D, D), dtype=object)
    factor = sp.Rational(1, Dr)
    kb = list(iproduct(*[range(d) for d in dk]))
    rb = list(iproduct(*[range(d) for d in dr]))
    for ik in kb:
        for jk in kb:
            ri = _flat(ik, dk); ci = _flat(jk, dk)
            cval = C_red[ri, ci]
            if cval == 0: continue
            term = factor * cval
            for ar in rb:
                row = [None]*n; col = [None]*n
                for p, k in enumerate(keep): row[k] = ik[p]; col[k] = jk[p]
                for p, k in enumerate(replace_idx): row[k] = ar[p]; col[k] = ar[p]
                result[_flat(row, dims), _flat(col, dims)] = term
    return result

def get_S3_permutation_matrix(perm):
    P = np.zeros((256, 256), dtype=object)
    for i in range(256):
        b = format(i, '08b')
        b_new = ''.join([b[perm[k]] for k in range(8)])
        P[int(b_new, 2), i] = 1
    return P

# =====================================================================
# 4. Main script: Extraction and deterministic solving
# =====================================================================
if __name__ == '__main__':
    T_op, T_op_dag = build_T_op_exact()

    P_12 = get_S3_permutation_matrix([0, 3, 4, 1, 2, 5, 6, 7])
    P_23 = get_S3_permutation_matrix([0, 1, 2, 5, 6, 3, 4, 7])

    print("\n" + "="*60)
    print(" -> Initializing optimization variables H0 to H8 (Schur blocks)...")
    blocks_info = [
        (4, 1), (6, 3), (2, 5),
        (6, 3), (9, 9), (3, 15),
        (2, 5), (3, 15), (1, 25)
    ]
    B_vars = [create_herm_matrix(f'H{i}', blocks_info[i][0]) for i in range(9)]

    M_sorted = np.zeros((256, 256), dtype=object)
    offset = 0
    for i, (p_dim, M_dim) in enumerate(blocks_info):
        size = p_dim * M_dim
        M_sorted[offset:offset+size, offset:offset+size] = np.kron(np.eye(M_dim, dtype=object), B_vars[i])
        offset += size

    C_np = T_op.dot(M_sorted).dot(T_op_dag)

    eq_list = []
    def extract_eqs(diff_mat, title):
        print(f"    - Extracting: {title}")
        for i in range(diff_mat.shape[0]):
            for j in range(i, diff_mat.shape[1]):
                val = sp.expand(diff_mat[i, j])
                if val != 0:
                    re, im = val.as_real_imag()
                    if re != 0: eq_list.append(re)
                    if im != 0: eq_list.append(im)

    print("\n" + "="*60)
    print("Executing sequential causal constraints...")
    start_t = time.time()

    extract_eqs(replace_subsystems_np(C_np, DIMS, [7]) - replace_subsystems_np(C_np, DIMS, [6, 7]), "Condition 1 (Tr_F)")
    extract_eqs(replace_subsystems_np(C_np, DIMS, [5, 6, 7]) - replace_subsystems_np(C_np, DIMS, [4, 5, 6, 7]), "Condition 2 (Tr_O3)")
    extract_eqs(replace_subsystems_np(C_np, DIMS, [3, 4, 5, 6, 7]) - replace_subsystems_np(C_np, DIMS, [2, 3, 4, 5, 6, 7]), "Condition 3 (Tr_O2)")
    extract_eqs(replace_subsystems_np(C_np, DIMS, [1, 2, 3, 4, 5, 6, 7]) - np.eye(256, dtype=object) * sp.Rational(1, 16), "Condition 4 (Global Trace)")

    print("\n" + "="*60)
    print("Injecting S3 symmetry constraints...")
    extract_eqs(C_np - P_12.dot(C_np).dot(P_12.T), "S3: Swap 1-2")
    extract_eqs(C_np - P_23.dot(C_np).dot(P_23.T), "S3: Swap 2-3")

    # Fix hash randomization by sorting equations and all variables deterministically
    unique_eqs_sorted = sorted(list(set(eq_list)), key=lambda x: str(x))
    all_symbols_sorted = sorted(list(real_symbols_set), key=lambda x: str(x))

    print(f"\nExtraction time: {time.time() - start_t:.2f} s")
    print(f"Extracted {len(unique_eqs_sorted)} real constraints, starting solver...")

    solve_t = time.time()
    # Solve for all variables using deterministic order
    solution = sp.solve(unique_eqs_sorted, all_symbols_sorted, dict=True)

    if solution:
        sol_dict = solution[0]
        free_vars = sorted([str(v) for v in (real_symbols_set - set(sol_dict.keys()))])
        print("\n" + "="*60)
        print(f"Solve successful! Final degrees of freedom: {len(free_vars)}")
        print("="*60)
        
        args = ", ".join([f"H{i}" for i in range(9)])
        print(f"\ndef apply_derived_constraints({args}):\n    constraints = []")
        
        def to_cvx(s):
            m = re.match(r'H(\d+)(R|I)?_(\d+)(\d+)', s)
            if not m: return s
            k, t, i, j = m.groups()
            return f"cp.{'imag' if t=='I' else 'real'}(H{k}[{i},{j}])"

        for var, expr in sol_dict.items():
            v_str = to_cvx(str(var))
            e_str = str(expr)
            # Replace symbol strings with CVXPY-friendly strings
            for fv_sym in all_symbols_sorted:
                fv_s = str(fv_sym)
                if fv_s in e_str:
                    e_str = re.sub(r'\b'+fv_s+r'\b', to_cvx(fv_s), e_str)
            print(f"    constraints.append({v_str} == {e_str})")
        print("    return constraints")
    else:
        print("No solution found!")


 -> Initializing optimization variables H0 to H8 (Schur blocks)...

Executing sequential causal constraints...
    - Extracting: Condition 1 (Tr_F)
    - Extracting: Condition 2 (Tr_O3)
    - Extracting: Condition 3 (Tr_O2)
    - Extracting: Condition 4 (Global Trace)

Injecting S3 symmetry constraints...
    - Extracting: S3: Swap 1-2
    - Extracting: S3: Swap 2-3

Extraction time: 774.17 s
Extracted 2754 real constraints, starting solver...

Solve successful! Final degrees of freedom: 25

def apply_derived_constraints(H0, H1, H2, H3, H4, H5, H6, H7, H8):
    constraints = []
    constraints.append(cp.imag(H0[0,1]) == 0)
    constraints.append(cp.imag(H0[0,2]) == 0)
    constraints.append(cp.imag(H0[0,3]) == 0)
    constraints.append(cp.imag(H0[1,2]) == 0)
    constraints.append(cp.imag(H0[1,3]) == 0)
    constraints.append(cp.imag(H0[2,3]) == 0)
    constraints.append(cp.real(H0[0,1]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[

In [2]:
import sympy as sp
from collections import defaultdict


class MyExpr:
    def __init__(self, expr):
        self.expr = expr
    def __add__(self, other):
        return MyExpr(self.expr + (other.expr if isinstance(other, MyExpr) else sp.sympify(other)))
    def __radd__(self, other):
        return MyExpr((other.expr if isinstance(other, MyExpr) else sp.sympify(other)) + self.expr)
    def __sub__(self, other):
        return MyExpr(self.expr - (other.expr if isinstance(other, MyExpr) else sp.sympify(other)))
    def __rsub__(self, other):
        return MyExpr((other.expr if isinstance(other, MyExpr) else sp.sympify(other)) - self.expr)
    def __mul__(self, other):
        return MyExpr(self.expr * (other.expr if isinstance(other, MyExpr) else sp.sympify(other)))
    def __rmul__(self, other):
        return MyExpr((other.expr if isinstance(other, MyExpr) else sp.sympify(other)) * self.expr)
    def __truediv__(self, other):
        return MyExpr(self.expr / (other.expr if isinstance(other, MyExpr) else sp.sympify(other)))
    def __neg__(self):
        return MyExpr(-self.expr)
    def __eq__(self, other):
        right = other.expr if isinstance(other, MyExpr) else sp.sympify(other)
        return self.expr - right

class MockCP:
    @staticmethod
    def real(var):
        if isinstance(var, (int, float)): return MyExpr(sp.sympify(var))
        return MyExpr(sp.Symbol(f"{var}_R"))
    @staticmethod
    def imag(var):
        if isinstance(var, (int, float)): return MyExpr(sp.sympify(0))
        return MyExpr(sp.Symbol(f"{var}_I"))

def sqrt(val):
    return MyExpr(sp.sqrt(val))

class MockMatrix:
    def __init__(self, name):
        self.name = name
    def __getitem__(self, idx):
        return f"{self.name}_{idx[0]}_{idx[1]}"

cp = MockCP()


def apply_derived_constraints(H0, H1, H2, H3, H4, H5, H6, H7, H8):
    constraints = []
    constraints.append(cp.imag(H0[0,1]) == 0)
    constraints.append(cp.imag(H0[0,2]) == 0)
    constraints.append(cp.imag(H0[0,3]) == 0)
    constraints.append(cp.imag(H0[1,2]) == 0)
    constraints.append(cp.imag(H0[1,3]) == 0)
    constraints.append(cp.imag(H0[2,3]) == 0)
    constraints.append(cp.real(H0[0,1]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[0,2]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[0,3]) == 3*cp.real(H1[1,3]) - 3*cp.real(H1[3,3]) + 3*cp.real(H1[4,4]))
    constraints.append(cp.real(H0[1,2]) == -3*cp.real(H1[1,3]))
    constraints.append(cp.real(H0[1,3]) == -3*sqrt(3)*cp.real(H1[3,3])/2 + 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[2,3]) == -3*sqrt(3)*cp.real(H1[3,3])/2 + 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[0,0]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.real(H0[1,1]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.real(H0[2,2]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.real(H0[3,3]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.imag(H1[0,1]) == 0)
    constraints.append(cp.imag(H1[0,2]) == -sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[0,3]) == 0)
    constraints.append(cp.imag(H1[0,4]) == 0)
    constraints.append(cp.imag(H1[0,5]) == cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[1,2]) == -cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[1,3]) == 0)
    constraints.append(cp.imag(H1[1,4]) == 0)
    constraints.append(cp.imag(H1[1,5]) == -sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[2,3]) == -cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[2,4]) == sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[2,5]) == 0)
    constraints.append(cp.imag(H1[3,4]) == 0)
    constraints.append(cp.imag(H1[3,5]) == sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.real(H1[0,1]) == -sqrt(3)*cp.real(H1[3,3])/2 + sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[0,2]) == -sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[0,3]) == -sqrt(3)*cp.real(H1[3,3])/2 + sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[0,4]) == -cp.real(H1[1,3]) + cp.real(H1[3,3]) - cp.real(H1[4,4]))
    constraints.append(cp.real(H1[0,5]) == cp.real(H1[4,5]))
    constraints.append(cp.real(H1[1,2]) == -cp.real(H1[4,5]))
    constraints.append(cp.real(H1[1,4]) == sqrt(3)*cp.real(H1[3,3])/2 - sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[1,5]) == -sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[2,3]) == cp.real(H1[4,5]))
    constraints.append(cp.real(H1[2,4]) == -sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[2,5]) == 0)
    constraints.append(cp.real(H1[3,4]) == sqrt(3)*cp.real(H1[3,3])/2 - sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[3,5]) == sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[0,0]) == cp.real(H1[4,4]))
    constraints.append(cp.real(H1[1,1]) == cp.real(H1[3,3]))
    constraints.append(cp.real(H1[2,2]) == -5*cp.real(H2[1,1])/3 - 9*cp.real(H4[5,5]) + 9*cp.real(H4[8,8])/2 - 15*cp.real(H5[1,1]) + 15*cp.real(H5[2,2])/2 - 5*cp.real(H7[2,2])/2 - 25*cp.real(H8[0,0])/6 + 4/3)
    constraints.append(cp.real(H1[5,5]) == -5*cp.real(H2[1,1])/3 - 9*cp.real(H4[5,5]) + 9*cp.real(H4[8,8])/2 - 15*cp.real(H5[1,1]) + 15*cp.real(H5[2,2])/2 - 5*cp.real(H7[2,2])/2 - 25*cp.real(H8[0,0])/6 + 4/3)
    constraints.append(cp.imag(H2[0,1]) == 0)
    constraints.append(cp.real(H2[0,1]) == 0)
    constraints.append(cp.real(H2[0,0]) == cp.real(H2[1,1]))
    constraints.append(cp.imag(H3[0,1]) == 0)
    constraints.append(cp.imag(H3[0,2]) == 0)
    constraints.append(cp.imag(H3[0,3]) == 4*sqrt(2)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[0,4]) == 3*sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[0,5]) == cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[1,2]) == 4*sqrt(2)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[1,3]) == 0)
    constraints.append(cp.imag(H3[1,4]) == cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[1,5]) == -3*sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[2,3]) == 0)
    constraints.append(cp.imag(H3[2,4]) == 3*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[2,5]) == -sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[3,4]) == -sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[3,5]) == -3*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[4,5]) == 0)
    constraints.append(cp.real(H3[0,1]) == 3*sqrt(3)*cp.real(H4[3,3])/2 - 3*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[6,6])/2 - 3*sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H3[0,2]) == -3*sqrt(3)*cp.real(H4[3,3])/2 - 9*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[5,5]) + 15*sqrt(3)*cp.real(H4[6,6])/4 + 9*sqrt(3)*cp.real(H4[7,7])/4 - 3*sqrt(3)*cp.real(H4[8,8]) + 5*sqrt(3)*cp.real(H5[1,1]) - 5*sqrt(3)*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[0,3]) == sqrt(6)*cp.real(H4[4,6])/2 - 3*cp.real(H4[3,3]) + 3*cp.real(H4[4,4]) - 69*cp.real(H4[6,6])/32 + 69*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H3[0,4]) == -3*sqrt(6)*cp.real(H4[3,3]) + 3*sqrt(6)*cp.real(H4[5,5])/2 + 15*sqrt(6)*cp.real(H4[6,6])/32 + 81*sqrt(6)*cp.real(H4[7,7])/32 - 3*sqrt(6)*cp.real(H4[8,8])/2 + 5*sqrt(6)*cp.real(H5[1,1])/2 - 5*sqrt(6)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[0,5]) == -sqrt(3)*cp.real(H4[4,6]) + 3*sqrt(2)*cp.real(H4[3,3]) - 3*sqrt(2)*cp.real(H4[4,4]) + 21*sqrt(2)*cp.real(H4[6,6])/16 - 21*sqrt(2)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H3[1,2]) == -sqrt(6)*cp.real(H4[4,6])/2 - 27*cp.real(H4[6,6])/32 + 27*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H3[1,3]) == -9*sqrt(3)*cp.real(H4[3,3])/2 - 3*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[5,5]) + 9*sqrt(3)*cp.real(H4[6,6])/4 + 15*sqrt(3)*cp.real(H4[7,7])/4 - 3*sqrt(3)*cp.real(H4[8,8]) + 5*sqrt(3)*cp.real(H5[1,1]) - 5*sqrt(3)*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[1,4]) == sqrt(3)*cp.real(H4[4,6]))
    constraints.append(cp.real(H3[1,5]) == -3*sqrt(6)*cp.real(H4[4,4]) + 3*sqrt(6)*cp.real(H4[5,5])/2 + 81*sqrt(6)*cp.real(H4[6,6])/32 + 15*sqrt(6)*cp.real(H4[7,7])/32 - 3*sqrt(6)*cp.real(H4[8,8])/2 + 5*sqrt(6)*cp.real(H5[1,1])/2 - 5*sqrt(6)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[2,3]) == -3*sqrt(3)*cp.real(H4[3,3])/2 + 3*sqrt(3)*cp.real(H4[4,4])/2 - 3*sqrt(3)*cp.real(H4[6,6])/2 + 3*sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H3[2,4]) == -3*sqrt(2)*cp.real(H4[3,3]) + 3*sqrt(2)*cp.real(H4[5,5])/2 + 15*sqrt(2)*cp.real(H4[6,6])/32 + 81*sqrt(2)*cp.real(H4[7,7])/32 - 3*sqrt(2)*cp.real(H4[8,8])/2 + 5*sqrt(2)*cp.real(H5[1,1])/2 - 5*sqrt(2)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[2,5]) == 3*cp.real(H4[4,6]) - 3*sqrt(6)*cp.real(H4[3,3]) + 3*sqrt(6)*cp.real(H4[4,4]) - 21*sqrt(6)*cp.real(H4[6,6])/16 + 21*sqrt(6)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H3[3,4]) == -3*cp.real(H4[4,6]))
    constraints.append(cp.real(H3[3,5]) == -3*sqrt(2)*cp.real(H4[4,4]) + 3*sqrt(2)*cp.real(H4[5,5])/2 + 81*sqrt(2)*cp.real(H4[6,6])/32 + 15*sqrt(2)*cp.real(H4[7,7])/32 - 3*sqrt(2)*cp.real(H4[8,8])/2 + 5*sqrt(2)*cp.real(H5[1,1])/2 - 5*sqrt(2)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[4,5]) == 0)
    constraints.append(cp.real(H3[0,0]) == -6*cp.real(H4[3,3]) - 9*cp.real(H4[4,4]) + 15*cp.real(H4[5,5])/2 + 15*cp.real(H4[6,6])/2 + 9*cp.real(H4[7,7])/2 - 6*cp.real(H4[8,8]) + 25*cp.real(H5[1,1])/2 - 10*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[1,1]) == -9*cp.real(H4[3,3]) - 6*cp.real(H4[4,4]) + 15*cp.real(H4[5,5])/2 + 9*cp.real(H4[6,6])/2 + 15*cp.real(H4[7,7])/2 - 6*cp.real(H4[8,8]) + 25*cp.real(H5[1,1])/2 - 10*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[2,2]) == -3*cp.real(H4[3,3]) + 3*cp.real(H4[5,5])/2 + 5*cp.real(H5[1,1])/2)
    constraints.append(cp.real(H3[3,3]) == -3*cp.real(H4[4,4]) + 3*cp.real(H4[5,5])/2 + 5*cp.real(H5[1,1])/2)
    constraints.append(cp.real(H3[4,4]) == -3*cp.real(H4[6,6]) + 3*cp.real(H4[8,8])/2 + 5*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[5,5]) == -3*cp.real(H4[7,7]) + 3*cp.real(H4[8,8])/2 + 5*cp.real(H5[2,2])/2)
    constraints.append(cp.imag(H4[0,1]) == 0)
    constraints.append(cp.imag(H4[0,2]) == 3*sqrt(2)*cp.imag(H4[5,6])/8 + sqrt(6)*cp.imag(H4[5,7])/8 - sqrt(3)*cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[0,3]) == 0)
    constraints.append(cp.imag(H4[0,4]) == -4*sqrt(2)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[0,5]) == -11*sqrt(6)*cp.imag(H4[5,6])/24 + 7*sqrt(2)*cp.imag(H4[5,7])/8 + cp.imag(H4[7,8]))
    constraints.append(cp.imag(H4[0,6]) == -sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H4[0,7]) == -cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[0,8]) == sqrt(3)*cp.imag(H4[5,6])/6 + cp.imag(H4[5,7])/2 + sqrt(2)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[1,2]) == sqrt(6)*cp.imag(H4[5,6])/8 + sqrt(2)*cp.imag(H4[5,7])/8 - cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[1,3]) == -4*sqrt(2)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[1,4]) == 0)
    constraints.append(cp.imag(H4[1,5]) == 3*sqrt(2)*cp.imag(H4[5,6])/8 + sqrt(6)*cp.imag(H4[5,7])/8)
    constraints.append(cp.imag(H4[1,6]) == -cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[1,7]) == sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H4[1,8]) == -3*cp.imag(H4[5,6])/2 + sqrt(3)*cp.imag(H4[5,7])/2 + 3*sqrt(6)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[2,3]) == -5*sqrt(6)*cp.imag(H4[5,6])/24 + 9*sqrt(2)*cp.imag(H4[5,7])/8)
    constraints.append(cp.imag(H4[2,4]) == -3*sqrt(2)*cp.imag(H4[5,6])/8 - sqrt(6)*cp.imag(H4[5,7])/8)
    constraints.append(cp.imag(H4[2,5]) == 0)
    constraints.append(cp.imag(H4[2,6]) == -sqrt(3)*cp.imag(H4[5,6])/3)
    constraints.append(cp.imag(H4[2,7]) == sqrt(3)*cp.imag(H4[5,7]))
    constraints.append(cp.imag(H4[2,8]) == 0)
    constraints.append(cp.imag(H4[3,4]) == 0)
    constraints.append(cp.imag(H4[3,5]) == -3*sqrt(2)*cp.imag(H4[5,6])/8 - sqrt(6)*cp.imag(H4[5,7])/8 + sqrt(3)*cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[3,6]) == -cp.imag(H4[4,7]))
    constraints.append(cp.imag(H4[3,7]) == sqrt(3)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[3,8]) == -cp.imag(H4[5,6])/2 - sqrt(3)*cp.imag(H4[5,7])/2 - sqrt(6)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[4,5]) == -sqrt(6)*cp.imag(H4[5,6])/8 - sqrt(2)*cp.imag(H4[5,7])/8 - cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[4,6]) == sqrt(3)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[4,8]) == -sqrt(3)*cp.imag(H4[5,6])/2 + cp.imag(H4[5,7])/2 + 3*sqrt(2)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[5,8]) == 0)
    constraints.append(cp.imag(H4[6,7]) == 0)
    constraints.append(cp.imag(H4[6,8]) == 0)
    constraints.append(cp.real(H4[0,1]) == -sqrt(3)*cp.real(H4[3,3])/2 + sqrt(3)*cp.real(H4[4,4])/2 - sqrt(3)*cp.real(H4[6,6])/2 + sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[0,2]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8 - sqrt(3)*cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[0,3]) == sqrt(3)*cp.real(H4[3,3])/2 + 3*sqrt(3)*cp.real(H4[4,4])/2 - 5*sqrt(3)*cp.real(H4[6,6])/4 - 3*sqrt(3)*cp.real(H4[7,7])/4)
    constraints.append(cp.real(H4[0,4]) == -sqrt(6)*cp.real(H4[4,6])/6 + cp.real(H4[3,3]) - cp.real(H4[4,4]) + 23*cp.real(H4[6,6])/32 - 23*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[0,5]) == 11*sqrt(6)*cp.real(H4[5,6])/24 - 7*sqrt(2)*cp.real(H4[5,7])/8 + cp.real(H4[7,8]))
    constraints.append(cp.real(H4[0,6]) == sqrt(6)*cp.real(H4[3,3]) - 5*sqrt(6)*cp.real(H4[6,6])/32 - 27*sqrt(6)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[0,7]) == sqrt(3)*cp.real(H4[4,6])/3 - sqrt(2)*cp.real(H4[3,3]) + sqrt(2)*cp.real(H4[4,4]) - 7*sqrt(2)*cp.real(H4[6,6])/16 + 7*sqrt(2)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H4[0,8]) == -sqrt(3)*cp.real(H4[5,6])/6 - cp.real(H4[5,7])/2 + sqrt(2)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[1,2]) == -sqrt(6)*cp.real(H4[5,6])/8 - sqrt(2)*cp.real(H4[5,7])/8 - cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[1,3]) == sqrt(6)*cp.real(H4[4,6])/6 + 9*cp.real(H4[6,6])/32 - 9*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[1,4]) == 3*sqrt(3)*cp.real(H4[3,3])/2 + sqrt(3)*cp.real(H4[4,4])/2 - 3*sqrt(3)*cp.real(H4[6,6])/4 - 5*sqrt(3)*cp.real(H4[7,7])/4)
    constraints.append(cp.real(H4[1,5]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8)
    constraints.append(cp.real(H4[1,6]) == -sqrt(3)*cp.real(H4[4,6])/3)
    constraints.append(cp.real(H4[1,7]) == sqrt(6)*cp.real(H4[4,4]) - 27*sqrt(6)*cp.real(H4[6,6])/32 - 5*sqrt(6)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[1,8]) == 3*cp.real(H4[5,6])/2 - sqrt(3)*cp.real(H4[5,7])/2 + 3*sqrt(6)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[2,3]) == -5*sqrt(6)*cp.real(H4[5,6])/24 + 9*sqrt(2)*cp.real(H4[5,7])/8)
    constraints.append(cp.real(H4[2,4]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8)
    constraints.append(cp.real(H4[2,5]) == 2*sqrt(3)*cp.real(H4[5,5]) - 2*sqrt(3)*cp.real(H4[8,8]))
    constraints.append(cp.real(H4[2,6]) == -sqrt(3)*cp.real(H4[5,6])/3)
    constraints.append(cp.real(H4[2,7]) == sqrt(3)*cp.real(H4[5,7]))
    constraints.append(cp.real(H4[2,8]) == sqrt(6)*cp.real(H4[5,5]) - sqrt(6)*cp.real(H4[8,8]))
    constraints.append(cp.real(H4[3,4]) == sqrt(3)*cp.real(H4[3,3])/2 - sqrt(3)*cp.real(H4[4,4])/2 + sqrt(3)*cp.real(H4[6,6])/2 - sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[3,5]) == 3*sqrt(2)*cp.real(H4[5,6])/8 + sqrt(6)*cp.real(H4[5,7])/8 + sqrt(3)*cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[3,6]) == sqrt(2)*cp.real(H4[3,3]) - 5*sqrt(2)*cp.real(H4[6,6])/32 - 27*sqrt(2)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[3,7]) == -cp.real(H4[4,6]) + sqrt(6)*cp.real(H4[3,3]) - sqrt(6)*cp.real(H4[4,4]) + 7*sqrt(6)*cp.real(H4[6,6])/16 - 7*sqrt(6)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H4[3,8]) == cp.real(H4[5,6])/2 + sqrt(3)*cp.real(H4[5,7])/2 - sqrt(6)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[4,5]) == sqrt(6)*cp.real(H4[5,6])/8 + sqrt(2)*cp.real(H4[5,7])/8 - cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[4,7]) == sqrt(2)*cp.real(H4[4,4]) - 27*sqrt(2)*cp.real(H4[6,6])/32 - 5*sqrt(2)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[4,8]) == sqrt(3)*cp.real(H4[5,6])/2 - cp.real(H4[5,7])/2 + 3*sqrt(2)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[5,8]) == sqrt(2)*cp.real(H4[5,5]) - sqrt(2)*cp.real(H4[8,8]))
    constraints.append(cp.real(H4[6,7]) == 0)
    constraints.append(cp.real(H4[6,8]) == 0)
    constraints.append(cp.real(H4[0,0]) == 2*cp.real(H4[3,3]) + 3*cp.real(H4[4,4]) - 5*cp.real(H4[6,6])/2 - 3*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[1,1]) == 3*cp.real(H4[3,3]) + 2*cp.real(H4[4,4]) - 3*cp.real(H4[6,6])/2 - 5*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[2,2]) == 5*cp.real(H4[5,5]) - 4*cp.real(H4[8,8]))
    constraints.append(cp.imag(H5[0,1]) == 0)
    constraints.append(cp.imag(H5[0,2]) == 0)
    constraints.append(cp.imag(H5[1,2]) == 0)
    constraints.append(cp.real(H5[0,1]) == 2*sqrt(3)*cp.real(H5[1,1]) - 2*sqrt(3)*cp.real(H5[2,2]))
    constraints.append(cp.real(H5[0,2]) == sqrt(6)*cp.real(H5[1,1]) - sqrt(6)*cp.real(H5[2,2]))
    constraints.append(cp.real(H5[1,2]) == sqrt(2)*cp.real(H5[1,1]) - sqrt(2)*cp.real(H5[2,2]))
    constraints.append(cp.real(H5[0,0]) == 5*cp.real(H5[1,1]) - 4*cp.real(H5[2,2]))
    constraints.append(cp.imag(H6[0,1]) == 0)
    constraints.append(cp.real(H6[0,1]) == 0)
    constraints.append(cp.real(H6[0,0]) == -3*cp.real(H7[1,1]) + 3*cp.real(H7[2,2])/2 + 5*cp.real(H8[0,0])/2)
    constraints.append(cp.real(H6[1,1]) == -3*cp.real(H7[1,1]) + 3*cp.real(H7[2,2])/2 + 5*cp.real(H8[0,0])/2)
    constraints.append(cp.imag(H7[0,1]) == 0)
    constraints.append(cp.imag(H7[0,2]) == 0)
    constraints.append(cp.imag(H7[1,2]) == 0)
    constraints.append(cp.real(H7[0,1]) == 0)
    constraints.append(cp.real(H7[0,2]) == 0)
    constraints.append(cp.real(H7[1,2]) == 0)
    constraints.append(cp.real(H7[0,0]) == cp.real(H7[1,1]))
    return constraints

# ==========================================
# System parsing and dependency pruning analysis (graph topology dimensionality reduction)
# ==========================================
def find_slack_variables():
    matrices = {name: MockMatrix(name) for name in ["H0","H1","H2","H3","H4","H5","H6","H7","H8"]}
    eqs = apply_derived_constraints(**matrices)
    
    # Build a bidirectional mapping graph from variables to equations
    var_to_eqs = defaultdict(set)
    for i, eq in enumerate(eqs):
        free_vars = eq.free_symbols
        for v in free_vars:
            var_to_eqs[v].add(i)
            
    pruned_eqs = set()
    pruned_vars = set()
    
    # Iteratively prune "leaf nodes" in the graph.
    # If a variable is only present in one unpruned equation, 
    # the equation simply defines it and has no downstream effect.
    changed = True
    while changed:
        changed = False
        leaf_vars = [v for v in var_to_eqs if v not in pruned_vars and len(var_to_eqs[v] - pruned_eqs) == 1]
        for v in leaf_vars:
            active_eqs = var_to_eqs[v] - pruned_eqs
            if not active_eqs: continue
            eq_idx = active_eqs.pop()
            
            pruned_vars.add(v)
            pruned_eqs.add(eq_idx)
            changed = True

    # Non-leaf variables with 0 associated equations after pruning are "Slack Variables".
    slack_vars = []
    for v in var_to_eqs:
        if v not in pruned_vars:
            active_eqs = var_to_eqs[v] - pruned_eqs
            if len(active_eqs) == 0:
                slack_vars.append(v)
                
    print("======== Slack Variables Analysis Report ========")
    print(f"Total variables: {len(var_to_eqs)}, Total equations: {len(eqs)}")
    print(f"Pruned dependent variables (leaf nodes): {len(pruned_vars)}")
    
        
    print("\n=> All extracted [Pure Slack Variables]:")
    slack_vars_sorted = sorted([str(v) for v in slack_vars])
    for sv in slack_vars_sorted:
        print(f"   - {sv}")
        
    # Check for core basis variables (neither defined leaves nor completely independent slack variables)
    core_vars = [v for v in var_to_eqs if v not in pruned_vars and len(var_to_eqs[v] - pruned_eqs) > 0]
    if not core_vars:
        print("\n=> Perfect dimensionality reduction! The constraint system forms a strict directed acyclic definition flow, and all remaining bases are completely independent.")

if __name__ == '__main__':
    find_slack_variables()

======== Slack Variables Analysis Report ========
Total variables: 196, Total equations: 171
Pruned dependent variables (leaf nodes): 171

=> All extracted [Pure Slack Variables]:
   - H1_1_3_R
   - H1_3_3_R
   - H1_4_4_R
   - H1_4_5_I
   - H1_4_5_R
   - H2_1_1_R
   - H4_3_3_R
   - H4_4_4_R
   - H4_4_6_R
   - H4_4_7_I
   - H4_5_5_R
   - H4_5_6_I
   - H4_5_6_R
   - H4_5_7_I
   - H4_5_7_R
   - H4_6_6_R
   - H4_7_7_R
   - H4_7_8_I
   - H4_7_8_R
   - H4_8_8_R
   - H5_1_1_R
   - H5_2_2_R
   - H7_1_1_R
   - H7_2_2_R
   - H8_0_0_R

=> Perfect dimensionality reduction! The constraint system forms a strict directed acyclic definition flow, and all remaining bases are completely independent.


In [ ]:
import sympy as sp
import re
import time

print("Starting to reveal the exact linear superposition relationship between constraints...")
start_time = time.time()

# ==========================================
# 1. Setup pure symbolic math environment
# ==========================================
def create_hermitian_sym_matrix(name, dim):
    """Create a Hermitian matrix with strict SymPy symbols"""
    M = sp.Matrix(dim, dim, lambda i, j: sp.Integer(0))
    for i in range(dim):
        M[i, i] = sp.Symbol(f'{name}R_{i}_{i}', real=True)
        for j in range(i+1, dim):
            r = sp.Symbol(f'{name}R_{i}_{j}', real=True)
            im = sp.Symbol(f'{name}I_{i}_{j}', real=True)
            M[i, j] = r + sp.I * im
            M[j, i] = r - sp.I * im  # Conjugate symmetry
    return M

H0 = create_hermitian_sym_matrix('H0', 4)
H1 = create_hermitian_sym_matrix('H1', 6)
H2 = create_hermitian_sym_matrix('H2', 2)
H3 = create_hermitian_sym_matrix('H3', 6)
H4 = create_hermitian_sym_matrix('H4', 9)
H5 = create_hermitian_sym_matrix('H5', 3)
H6 = create_hermitian_sym_matrix('H6', 2)
H7 = create_hermitian_sym_matrix('H7', 3)
H8 = create_hermitian_sym_matrix('H8', 1)

class MockCP:
    @staticmethod
    def real(val): return sp.re(val)
    @staticmethod
    def imag(val): return sp.im(val)

cp = MockCP()
sqrt = sp.sqrt

# ==========================================
# 2. Mount and parse raw constraints of Program 1 and and fix some redundant variables in Cell 2
# ==========================================
raw_prog1 = """
constraints.append(cp.imag(H0[0,1]) == 0)
constraints.append(cp.imag(H0[0,2]) == 0)
constraints.append(cp.imag(H0[0,3]) == 0)
constraints.append(cp.imag(H0[1,2]) == 0)
constraints.append(cp.imag(H0[1,3]) == 0)
constraints.append(cp.imag(H0[2,3]) == 0)
constraints.append(cp.real(H0[0,1]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H0[0,2]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H0[0,3]) == 3*cp.real(H1[1,3]) - 3*cp.real(H1[3,3]) + 3*cp.real(H1[4,4]))
constraints.append(cp.real(H0[1,2]) == -3*cp.real(H1[1,3]))
constraints.append(cp.real(H0[1,3]) == -3*sqrt(3)*cp.real(H1[3,3])/2 + 3*sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H0[2,3]) == -3*sqrt(3)*cp.real(H1[3,3])/2 + 3*sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H0[0,0]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
constraints.append(cp.real(H0[1,1]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
constraints.append(cp.real(H0[2,2]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
constraints.append(cp.real(H0[3,3]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
constraints.append(cp.imag(H1[0,1]) == 0)
constraints.append(cp.imag(H1[0,2]) == -sqrt(3)*cp.imag(H1[4,5]))
constraints.append(cp.imag(H1[0,3]) == 0)
constraints.append(cp.imag(H1[0,4]) == 0)
constraints.append(cp.imag(H1[0,5]) == cp.imag(H1[4,5]))
constraints.append(cp.imag(H1[1,2]) == -cp.imag(H1[4,5]))
constraints.append(cp.imag(H1[1,3]) == 0)
constraints.append(cp.imag(H1[1,4]) == 0)
constraints.append(cp.imag(H1[1,5]) == -sqrt(3)*cp.imag(H1[4,5]))
constraints.append(cp.imag(H1[2,3]) == -cp.imag(H1[4,5]))
constraints.append(cp.imag(H1[2,4]) == sqrt(3)*cp.imag(H1[4,5]))
constraints.append(cp.imag(H1[2,5]) == 0)
constraints.append(cp.imag(H1[3,4]) == 0)
constraints.append(cp.imag(H1[3,5]) == sqrt(3)*cp.imag(H1[4,5]))
constraints.append(cp.real(H1[0,1]) == -sqrt(3)*cp.real(H1[3,3])/2 + sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H1[0,2]) == -sqrt(3)*cp.real(H1[4,5]))
constraints.append(cp.real(H1[0,3]) == -sqrt(3)*cp.real(H1[3,3])/2 + sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H1[0,4]) == -cp.real(H1[1,3]) + cp.real(H1[3,3]) - cp.real(H1[4,4]))
constraints.append(cp.real(H1[0,5]) == cp.real(H1[4,5]))
constraints.append(cp.real(H1[1,2]) == -cp.real(H1[4,5]))
constraints.append(cp.real(H1[1,4]) == sqrt(3)*cp.real(H1[3,3])/2 - sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H1[1,5]) == -sqrt(3)*cp.real(H1[4,5]))
constraints.append(cp.real(H1[2,3]) == cp.real(H1[4,5]))
constraints.append(cp.real(H1[2,4]) == -sqrt(3)*cp.real(H1[4,5]))
constraints.append(cp.real(H1[2,5]) == 0)
constraints.append(cp.real(H1[3,4]) == sqrt(3)*cp.real(H1[3,3])/2 - sqrt(3)*cp.real(H1[4,4])/2)
constraints.append(cp.real(H1[3,5]) == sqrt(3)*cp.real(H1[4,5]))
constraints.append(cp.real(H1[0,0]) == cp.real(H1[4,4]))
constraints.append(cp.real(H1[1,1]) == cp.real(H1[3,3]))
constraints.append(cp.real(H1[2,2]) == -5*cp.real(H2[1,1])/3 - 9*cp.real(H4[5,5]) + 9*cp.real(H4[8,8])/2 - 15*cp.real(H5[1,1]) + 15*cp.real(H5[2,2])/2 - 5*cp.real(H7[2,2])/2 - 25*cp.real(H8[0,0])/6 + 4/3)
constraints.append(cp.real(H1[5,5]) == -5*cp.real(H2[1,1])/3 - 9*cp.real(H4[5,5]) + 9*cp.real(H4[8,8])/2 - 15*cp.real(H5[1,1]) + 15*cp.real(H5[2,2])/2 - 5*cp.real(H7[2,2])/2 - 25*cp.real(H8[0,0])/6 + 4/3)
constraints.append(cp.imag(H2[0,1]) == 0)
constraints.append(cp.real(H2[0,1]) == 0)
constraints.append(cp.real(H2[0,0]) == cp.real(H2[1,1]))
constraints.append(cp.imag(H3[0,1]) == 0)
constraints.append(cp.imag(H3[0,2]) == 0)
constraints.append(cp.imag(H3[0,3]) == 4*sqrt(2)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[0,4]) == 3*sqrt(3)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[0,5]) == cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[1,2]) == 4*sqrt(2)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[1,3]) == 0)
constraints.append(cp.imag(H3[1,4]) == cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[1,5]) == -3*sqrt(3)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[2,3]) == 0)
constraints.append(cp.imag(H3[2,4]) == 3*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[2,5]) == -sqrt(3)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[3,4]) == -sqrt(3)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[3,5]) == -3*cp.imag(H4[4,7]))
constraints.append(cp.imag(H3[4,5]) == 0)
constraints.append(cp.real(H3[0,1]) == 3*sqrt(3)*cp.real(H4[3,3])/2 - 3*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[6,6])/2 - 3*sqrt(3)*cp.real(H4[7,7])/2)
constraints.append(cp.real(H3[0,2]) == -3*sqrt(3)*cp.real(H4[3,3])/2 - 9*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[5,5]) + 15*sqrt(3)*cp.real(H4[6,6])/4 + 9*sqrt(3)*cp.real(H4[7,7])/4 - 3*sqrt(3)*cp.real(H4[8,8]) + 5*sqrt(3)*cp.real(H5[1,1]) - 5*sqrt(3)*cp.real(H5[2,2]))
constraints.append(cp.real(H3[0,3]) == sqrt(6)*cp.real(H4[4,6])/2 - 3*cp.real(H4[3,3]) + 3*cp.real(H4[4,4]) - 69*cp.real(H4[6,6])/32 + 69*cp.real(H4[7,7])/32)
constraints.append(cp.real(H3[0,4]) == -3*sqrt(6)*cp.real(H4[3,3]) + 3*sqrt(6)*cp.real(H4[5,5])/2 + 15*sqrt(6)*cp.real(H4[6,6])/32 + 81*sqrt(6)*cp.real(H4[7,7])/32 - 3*sqrt(6)*cp.real(H4[8,8])/2 + 5*sqrt(6)*cp.real(H5[1,1])/2 - 5*sqrt(6)*cp.real(H5[2,2])/2)
constraints.append(cp.real(H3[0,5]) == -sqrt(3)*cp.real(H4[4,6]) + 3*sqrt(2)*cp.real(H4[3,3]) - 3*sqrt(2)*cp.real(H4[4,4]) + 21*sqrt(2)*cp.real(H4[6,6])/16 - 21*sqrt(2)*cp.real(H4[7,7])/16)
constraints.append(cp.real(H3[1,2]) == -sqrt(6)*cp.real(H4[4,6])/2 - 27*cp.real(H4[6,6])/32 + 27*cp.real(H4[7,7])/32)
constraints.append(cp.real(H3[1,3]) == -9*sqrt(3)*cp.real(H4[3,3])/2 - 3*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[5,5]) + 9*sqrt(3)*cp.real(H4[6,6])/4 + 15*sqrt(3)*cp.real(H4[7,7])/4 - 3*sqrt(3)*cp.real(H4[8,8]) + 5*sqrt(3)*cp.real(H5[1,1]) - 5*sqrt(3)*cp.real(H5[2,2]))
constraints.append(cp.real(H3[1,4]) == sqrt(3)*cp.real(H4[4,6]))
constraints.append(cp.real(H3[1,5]) == -3*sqrt(6)*cp.real(H4[4,4]) + 3*sqrt(6)*cp.real(H4[5,5])/2 + 81*sqrt(6)*cp.real(H4[6,6])/32 + 15*sqrt(6)*cp.real(H4[7,7])/32 - 3*sqrt(6)*cp.real(H4[8,8])/2 + 5*sqrt(6)*cp.real(H5[1,1])/2 - 5*sqrt(6)*cp.real(H5[2,2])/2)
constraints.append(cp.real(H3[2,3]) == -3*sqrt(3)*cp.real(H4[3,3])/2 + 3*sqrt(3)*cp.real(H4[4,4])/2 - 3*sqrt(3)*cp.real(H4[6,6])/2 + 3*sqrt(3)*cp.real(H4[7,7])/2)
constraints.append(cp.real(H3[2,4]) == -3*sqrt(2)*cp.real(H4[3,3]) + 3*sqrt(2)*cp.real(H4[5,5])/2 + 15*sqrt(2)*cp.real(H4[6,6])/32 + 81*sqrt(2)*cp.real(H4[7,7])/32 - 3*sqrt(2)*cp.real(H4[8,8])/2 + 5*sqrt(2)*cp.real(H5[1,1])/2 - 5*sqrt(2)*cp.real(H5[2,2])/2)
constraints.append(cp.real(H3[2,5]) == 3*cp.real(H4[4,6]) - 3*sqrt(6)*cp.real(H4[3,3]) + 3*sqrt(6)*cp.real(H4[4,4]) - 21*sqrt(6)*cp.real(H4[6,6])/16 + 21*sqrt(6)*cp.real(H4[7,7])/16)
constraints.append(cp.real(H3[3,4]) == -3*cp.real(H4[4,6]))
constraints.append(cp.real(H3[3,5]) == -3*sqrt(2)*cp.real(H4[4,4]) + 3*sqrt(2)*cp.real(H4[5,5])/2 + 81*sqrt(2)*cp.real(H4[6,6])/32 + 15*sqrt(2)*cp.real(H4[7,7])/32 - 3*sqrt(2)*cp.real(H4[8,8])/2 + 5*sqrt(2)*cp.real(H5[1,1])/2 - 5*sqrt(2)*cp.real(H5[2,2])/2)
constraints.append(cp.real(H3[4,5]) == 0)
constraints.append(cp.real(H3[0,0]) == -6*cp.real(H4[3,3]) - 9*cp.real(H4[4,4]) + 15*cp.real(H4[5,5])/2 + 15*cp.real(H4[6,6])/2 + 9*cp.real(H4[7,7])/2 - 6*cp.real(H4[8,8]) + 25*cp.real(H5[1,1])/2 - 10*cp.real(H5[2,2]))
constraints.append(cp.real(H3[1,1]) == -9*cp.real(H4[3,3]) - 6*cp.real(H4[4,4]) + 15*cp.real(H4[5,5])/2 + 9*cp.real(H4[6,6])/2 + 15*cp.real(H4[7,7])/2 - 6*cp.real(H4[8,8]) + 25*cp.real(H5[1,1])/2 - 10*cp.real(H5[2,2]))
constraints.append(cp.real(H3[2,2]) == -3*cp.real(H4[3,3]) + 3*cp.real(H4[5,5])/2 + 5*cp.real(H5[1,1])/2)
constraints.append(cp.real(H3[3,3]) == -3*cp.real(H4[4,4]) + 3*cp.real(H4[5,5])/2 + 5*cp.real(H5[1,1])/2)
constraints.append(cp.real(H3[4,4]) == -3*cp.real(H4[6,6]) + 3*cp.real(H4[8,8])/2 + 5*cp.real(H5[2,2])/2)
constraints.append(cp.real(H3[5,5]) == -3*cp.real(H4[7,7]) + 3*cp.real(H4[8,8])/2 + 5*cp.real(H5[2,2])/2)
constraints.append(cp.imag(H4[0,1]) == 0)
constraints.append(cp.imag(H4[0,2]) == 3*sqrt(2)*cp.imag(H4[5,6])/8 + sqrt(6)*cp.imag(H4[5,7])/8 - sqrt(3)*cp.imag(H4[7,8])/2)
constraints.append(cp.imag(H4[0,3]) == 0)
constraints.append(cp.imag(H4[0,4]) == -4*sqrt(2)*cp.imag(H4[4,7])/3)
constraints.append(cp.imag(H4[0,5]) == -11*sqrt(6)*cp.imag(H4[5,6])/24 + 7*sqrt(2)*cp.imag(H4[5,7])/8 + cp.imag(H4[7,8]))
constraints.append(cp.imag(H4[0,6]) == -sqrt(3)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H4[0,7]) == -cp.imag(H4[4,7])/3)
constraints.append(cp.imag(H4[0,8]) == sqrt(3)*cp.imag(H4[5,6])/6 + cp.imag(H4[5,7])/2 + sqrt(2)*cp.imag(H4[7,8])/8)
constraints.append(cp.imag(H4[1,2]) == sqrt(6)*cp.imag(H4[5,6])/8 + sqrt(2)*cp.imag(H4[5,7])/8 - cp.imag(H4[7,8])/2)
constraints.append(cp.imag(H4[1,3]) == -4*sqrt(2)*cp.imag(H4[4,7])/3)
constraints.append(cp.imag(H4[1,4]) == 0)
constraints.append(cp.imag(H4[1,5]) == 3*sqrt(2)*cp.imag(H4[5,6])/8 + sqrt(6)*cp.imag(H4[5,7])/8)
constraints.append(cp.imag(H4[1,6]) == -cp.imag(H4[4,7])/3)
constraints.append(cp.imag(H4[1,7]) == sqrt(3)*cp.imag(H4[4,7]))
constraints.append(cp.imag(H4[1,8]) == -3*cp.imag(H4[5,6])/2 + sqrt(3)*cp.imag(H4[5,7])/2 + 3*sqrt(6)*cp.imag(H4[7,8])/8)
constraints.append(cp.imag(H4[2,3]) == -5*sqrt(6)*cp.imag(H4[5,6])/24 + 9*sqrt(2)*cp.imag(H4[5,7])/8)
constraints.append(cp.imag(H4[2,4]) == -3*sqrt(2)*cp.imag(H4[5,6])/8 - sqrt(6)*cp.imag(H4[5,7])/8)
constraints.append(cp.imag(H4[2,5]) == 0)
constraints.append(cp.imag(H4[2,6]) == -sqrt(3)*cp.imag(H4[5,6])/3)
constraints.append(cp.imag(H4[2,7]) == sqrt(3)*cp.imag(H4[5,7]))
constraints.append(cp.imag(H4[2,8]) == 0)
constraints.append(cp.imag(H4[3,4]) == 0)
constraints.append(cp.imag(H4[3,5]) == -3*sqrt(2)*cp.imag(H4[5,6])/8 - sqrt(6)*cp.imag(H4[5,7])/8 + sqrt(3)*cp.imag(H4[7,8])/2)
constraints.append(cp.imag(H4[3,6]) == -cp.imag(H4[4,7]))
constraints.append(cp.imag(H4[3,7]) == sqrt(3)*cp.imag(H4[4,7])/3)
constraints.append(cp.imag(H4[3,8]) == -cp.imag(H4[5,6])/2 - sqrt(3)*cp.imag(H4[5,7])/2 - sqrt(6)*cp.imag(H4[7,8])/8)
constraints.append(cp.imag(H4[4,5]) == -sqrt(6)*cp.imag(H4[5,6])/8 - sqrt(2)*cp.imag(H4[5,7])/8 - cp.imag(H4[7,8])/2)
constraints.append(cp.imag(H4[4,6]) == sqrt(3)*cp.imag(H4[4,7])/3)
constraints.append(cp.imag(H4[4,8]) == -sqrt(3)*cp.imag(H4[5,6])/2 + cp.imag(H4[5,7])/2 + 3*sqrt(2)*cp.imag(H4[7,8])/8)
constraints.append(cp.imag(H4[5,8]) == 0)
constraints.append(cp.imag(H4[6,7]) == 0)
constraints.append(cp.imag(H4[6,8]) == 0)
constraints.append(cp.real(H4[0,1]) == -sqrt(3)*cp.real(H4[3,3])/2 + sqrt(3)*cp.real(H4[4,4])/2 - sqrt(3)*cp.real(H4[6,6])/2 + sqrt(3)*cp.real(H4[7,7])/2)
constraints.append(cp.real(H4[0,2]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8 - sqrt(3)*cp.real(H4[7,8])/2)
constraints.append(cp.real(H4[0,3]) == sqrt(3)*cp.real(H4[3,3])/2 + 3*sqrt(3)*cp.real(H4[4,4])/2 - 5*sqrt(3)*cp.real(H4[6,6])/4 - 3*sqrt(3)*cp.real(H4[7,7])/4)
constraints.append(cp.real(H4[0,4]) == -sqrt(6)*cp.real(H4[4,6])/6 + cp.real(H4[3,3]) - cp.real(H4[4,4]) + 23*cp.real(H4[6,6])/32 - 23*cp.real(H4[7,7])/32)
constraints.append(cp.real(H4[0,5]) == 11*sqrt(6)*cp.real(H4[5,6])/24 - 7*sqrt(2)*cp.real(H4[5,7])/8 + cp.real(H4[7,8]))
constraints.append(cp.real(H4[0,6]) == sqrt(6)*cp.real(H4[3,3]) - 5*sqrt(6)*cp.real(H4[6,6])/32 - 27*sqrt(6)*cp.real(H4[7,7])/32)
constraints.append(cp.real(H4[0,7]) == sqrt(3)*cp.real(H4[4,6])/3 - sqrt(2)*cp.real(H4[3,3]) + sqrt(2)*cp.real(H4[4,4]) - 7*sqrt(2)*cp.real(H4[6,6])/16 + 7*sqrt(2)*cp.real(H4[7,7])/16)
constraints.append(cp.real(H4[0,8]) == -sqrt(3)*cp.real(H4[5,6])/6 - cp.real(H4[5,7])/2 + sqrt(2)*cp.real(H4[7,8])/8)
constraints.append(cp.real(H4[1,2]) == -sqrt(6)*cp.real(H4[5,6])/8 - sqrt(2)*cp.real(H4[5,7])/8 - cp.real(H4[7,8])/2)
constraints.append(cp.real(H4[1,3]) == sqrt(6)*cp.real(H4[4,6])/6 + 9*cp.real(H4[6,6])/32 - 9*cp.real(H4[7,7])/32)
constraints.append(cp.real(H4[1,4]) == 3*sqrt(3)*cp.real(H4[3,3])/2 + sqrt(3)*cp.real(H4[4,4])/2 - 3*sqrt(3)*cp.real(H4[6,6])/4 - 5*sqrt(3)*cp.real(H4[7,7])/4)
constraints.append(cp.real(H4[1,5]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8)
constraints.append(cp.real(H4[1,6]) == -sqrt(3)*cp.real(H4[4,6])/3)
constraints.append(cp.real(H4[1,7]) == sqrt(6)*cp.real(H4[4,4]) - 27*sqrt(6)*cp.real(H4[6,6])/32 - 5*sqrt(6)*cp.real(H4[7,7])/32)
constraints.append(cp.real(H4[1,8]) == 3*cp.real(H4[5,6])/2 - sqrt(3)*cp.real(H4[5,7])/2 + 3*sqrt(6)*cp.real(H4[7,8])/8)
constraints.append(cp.real(H4[2,3]) == -5*sqrt(6)*cp.real(H4[5,6])/24 + 9*sqrt(2)*cp.real(H4[5,7])/8)
constraints.append(cp.real(H4[2,4]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8)
constraints.append(cp.real(H4[2,5]) == 2*sqrt(3)*cp.real(H4[5,5]) - 2*sqrt(3)*cp.real(H4[8,8]))
constraints.append(cp.real(H4[2,6]) == -sqrt(3)*cp.real(H4[5,6])/3)
constraints.append(cp.real(H4[2,7]) == sqrt(3)*cp.real(H4[5,7]))
constraints.append(cp.real(H4[2,8]) == sqrt(6)*cp.real(H4[5,5]) - sqrt(6)*cp.real(H4[8,8]))
constraints.append(cp.real(H4[3,4]) == sqrt(3)*cp.real(H4[3,3])/2 - sqrt(3)*cp.real(H4[4,4])/2 + sqrt(3)*cp.real(H4[6,6])/2 - sqrt(3)*cp.real(H4[7,7])/2)
constraints.append(cp.real(H4[3,5]) == 3*sqrt(2)*cp.real(H4[5,6])/8 + sqrt(6)*cp.real(H4[5,7])/8 + sqrt(3)*cp.real(H4[7,8])/2)
constraints.append(cp.real(H4[3,6]) == sqrt(2)*cp.real(H4[3,3]) - 5*sqrt(2)*cp.real(H4[6,6])/32 - 27*sqrt(2)*cp.real(H4[7,7])/32)
constraints.append(cp.real(H4[3,7]) == -cp.real(H4[4,6]) + sqrt(6)*cp.real(H4[3,3]) - sqrt(6)*cp.real(H4[4,4]) + 7*sqrt(6)*cp.real(H4[6,6])/16 - 7*sqrt(6)*cp.real(H4[7,7])/16)
constraints.append(cp.real(H4[3,8]) == cp.real(H4[5,6])/2 + sqrt(3)*cp.real(H4[5,7])/2 - sqrt(6)*cp.real(H4[7,8])/8)
constraints.append(cp.real(H4[4,5]) == sqrt(6)*cp.real(H4[5,6])/8 + sqrt(2)*cp.real(H4[5,7])/8 - cp.real(H4[7,8])/2)
constraints.append(cp.real(H4[4,7]) == sqrt(2)*cp.real(H4[4,4]) - 27*sqrt(2)*cp.real(H4[6,6])/32 - 5*sqrt(2)*cp.real(H4[7,7])/32)
constraints.append(cp.real(H4[4,8]) == sqrt(3)*cp.real(H4[5,6])/2 - cp.real(H4[5,7])/2 + 3*sqrt(2)*cp.real(H4[7,8])/8)
constraints.append(cp.real(H4[5,8]) == sqrt(2)*cp.real(H4[5,5]) - sqrt(2)*cp.real(H4[8,8]))
constraints.append(cp.real(H4[6,7]) == 0)
constraints.append(cp.real(H4[6,8]) == 0)
constraints.append(cp.real(H4[0,0]) == 2*cp.real(H4[3,3]) + 3*cp.real(H4[4,4]) - 5*cp.real(H4[6,6])/2 - 3*cp.real(H4[7,7])/2)
constraints.append(cp.real(H4[1,1]) == 3*cp.real(H4[3,3]) + 2*cp.real(H4[4,4]) - 3*cp.real(H4[6,6])/2 - 5*cp.real(H4[7,7])/2)
constraints.append(cp.real(H4[2,2]) == 5*cp.real(H4[5,5]) - 4*cp.real(H4[8,8]))
constraints.append(cp.imag(H5[0,1]) == 0)
constraints.append(cp.imag(H5[0,2]) == 0)
constraints.append(cp.imag(H5[1,2]) == 0)
constraints.append(cp.real(H5[0,1]) == 2*sqrt(3)*cp.real(H5[1,1]) - 2*sqrt(3)*cp.real(H5[2,2]))
constraints.append(cp.real(H5[0,2]) == sqrt(6)*cp.real(H5[1,1]) - sqrt(6)*cp.real(H5[2,2]))
constraints.append(cp.real(H5[1,2]) == sqrt(2)*cp.real(H5[1,1]) - sqrt(2)*cp.real(H5[2,2]))
constraints.append(cp.real(H5[0,0]) == 5*cp.real(H5[1,1]) - 4*cp.real(H5[2,2]))
constraints.append(cp.imag(H6[0,1]) == 0)
constraints.append(cp.real(H6[0,1]) == 0)
constraints.append(cp.real(H6[0,0]) == -3*cp.real(H7[1,1]) + 3*cp.real(H7[2,2])/2 + 5*cp.real(H8[0,0])/2)
constraints.append(cp.real(H6[1,1]) == -3*cp.real(H7[1,1]) + 3*cp.real(H7[2,2])/2 + 5*cp.real(H8[0,0])/2)
constraints.append(cp.imag(H7[0,1]) == 0)
constraints.append(cp.imag(H7[0,2]) == 0)
constraints.append(cp.imag(H7[1,2]) == 0)
constraints.append(cp.real(H7[0,1]) == 0)
constraints.append(cp.real(H7[0,2]) == 0)
constraints.append(cp.real(H7[1,2]) == 0)
constraints.append(cp.real(H7[0,0]) == cp.real(H7[1,1]))
constraints.append(cp.real(H2[1,1]) == 0)
constraints.append(cp.real(H4[5,5]) == 0)
constraints.append(cp.real(H4[8,8]) == 0)
constraints.append(cp.real(H5[1,1]) == 0)
constraints.append(cp.real(H5[2,2]) == 0)
constraints.append(cp.real(H7[2,2]) == 0)
constraints.append(cp.real(H8[0,0]) == 0)
"""
lines_prog1 = [line.strip() for line in raw_prog1.strip().split('\n') if line.strip()]
eqs_prog1 = []
class ConstraintsProg1:
    def append(self, val): eqs_prog1.append(sp.nsimplify(val, rational=True))
constraints = ConstraintsProg1()
exec(re.sub(r'==\s*(.*)\)', r'- (\1))', raw_prog1))

# ==========================================
# 3. Construct projection matrix & solve Program 1 space
# ==========================================
preferred_vars = {

}

def var_priority(var):
    """Custom elimination priority: lower number = higher priority to eliminate"""
    name = var.name
    
    if name in preferred_vars:
        return 10 
        
    if 'H0' in name or 'H1' in name: return 0 
    if 'H4' in name or 'H5' in name or 'H6' in name or 'H7' in name or 'H8' in name: return 1 
    if 'H3' in name: return 2
    if 'H2' in name: return 3 
    return 4

all_vars = list(set().union(*[eq.free_symbols for eq in eqs_prog1 if hasattr(eq, 'free_symbols')]))
all_vars = sorted(list(all_vars), key=lambda x: (var_priority(x), x.name))

A1, b1 = sp.linear_eq_to_matrix(eqs_prog1, all_vars)
M_matrix = A1.row_join(b1).T  # Transpose: (N+1) x M

print(f"Constraint space established...")
print("Solving general solution space for Program 1...")

sol_prog1 = sp.linsolve((A1, b1), all_vars)
subs_prog1 = {}
if sol_prog1:
    sol_prog1_tuple = list(sol_prog1)[0]
    subs_prog1 = {var: val for var, val in zip(all_vars, sol_prog1_tuple)}

# ==========================================
# 4. Parse Program 2 and calculate superposition coefficients
# ==========================================
raw_prog2 = """
constraints2.append(H1[1,1] == H0[0,0]/3 - H0[1,1]/3 + H1[0,0])
constraints2.append(H1[2,2] == 2*H0[0,0]/3 + 2*H1[0,0])
constraints2.append(H1[3,3] == -H0[0,0]/3 - H0[2,2]/3 - H1[0,0] + 4/3)
constraints2.append(H1[4,4] == -H0[0,0]/3 - H0[3,3]/3 - H1[0,0] + 4/3)
constraints2.append(H1[5,5] == -2*H0[0,0]/3 - 2*H1[0,0] + 8/3)
constraints2.append(H1[0,1] == -H0[0,1]/3)
constraints2.append(H1[0,4] == -H0[0,3]/3)
constraints2.append(H1[1,3] == -H0[1,2]/3)
constraints2.append(H1[3,4] == -H0[2,3]/3)
constraints2.append(H1[1,4] == H1[2,5]/2 - H0[1,3]/3)
constraints2.append(H1[0,3] == H1[2,5]/2 - H0[0,2]/3)
"""
lines_prog2 = [line.strip().replace('constraints2.append(', '').replace(')', '') 
               for line in raw_prog2.strip().split('\n') if line.strip()]
eqs_prog2_complex = []
class ConstraintsProg2:
    def append(self, val): eqs_prog2_complex.append(sp.nsimplify(val, rational=True))
constraints2 = ConstraintsProg2()
exec(re.sub(r'==\s*(.*)\)', r'- (\1))', raw_prog2))

print("\n ============ Algebraic Perspective Results ==============")
for i, eq_expr in enumerate(eqs_prog2_complex):
    print(f"\n Target Program 2, Line {i+1}: {lines_prog2[i]}")
    
    parts = []
    eq_expr_exp = sp.expand(eq_expr)
    eq_re, eq_im = eq_expr_exp.as_real_imag()
    
    if eq_re != 0: parts.append(('Real Part', sp.simplify(eq_re)))
    if eq_im != 0: parts.append(('Imaginary Part', sp.simplify(eq_im)))
        
    for part_name, part_expr in parts:
        print(f"\n  For the disassembled [{part_name}] constraint:")
        
        A2, b2 = sp.linear_eq_to_matrix([part_expr], all_vars)
        target_vec = A2.row_join(b2).T 
        
        system = M_matrix.row_join(target_vec)
        sol = sp.linsolve(system)
        
        if not sol:
            print("     Cannot match perfectly!")
            
            # 1. Extract leftover algebraic requirements
            leftover_expr = sp.simplify(part_expr.subs(subs_prog1))
            
            # 2. Extract matched parts from Program 1
            matched_expr = sp.simplify(part_expr - leftover_expr)
            
            if matched_expr != 0:
                A_match, b_match = sp.linear_eq_to_matrix([matched_expr], all_vars)
                target_match = A_match.row_join(b_match).T
                sol_match = sp.linsolve(M_matrix.row_join(target_match))
                
                if sol_match:
                    print("     Matched identical parts from Program 1:")
                    sol_tuple = list(sol_match)[0]
                    subs_dict = {sym: sp.S.Zero for sym in sol_tuple.free_symbols}
                    for orig_idx, coeff in enumerate(sol_tuple):
                        val = sp.simplify(coeff.subs(subs_dict))
                        if val != 0:
                            src_line = lines_prog1[orig_idx]
                            if src_line.startswith('constraints.append('):
                                src_line = src_line[19:-1] 
                            print(f"        Coefficient [ {str(val):^5} ] * Prog 1 Line {orig_idx+1:02d} -> {src_line}")
            else:
                print("    (Target constraint is completely unrelated to Program 1)")

            # 3. Identify conflicts or leftover requirements
            if leftover_expr.is_number:
                print(f"     Conflict: Variables matched, but a constant remains! -> {leftover_expr} != 0")
            else:
                print(f"     Leftover requirement: To be valid, must also satisfy -> {leftover_expr} == 0")

        else:
            sol_tuple = list(sol)[0]
            subs_dict = {sym: sp.S.Zero for sym in sol_tuple.free_symbols}
            
            found = False
            for orig_idx, coeff in enumerate(sol_tuple):
                val = sp.simplify(coeff.subs(subs_dict))
                if val != 0:
                    src_line = lines_prog1[orig_idx]
                    if src_line.startswith('constraints.append('):
                        src_line = src_line[19:-1] 
                    print(f"     Coefficient [ {str(val):^5} ] * Prog 1 Line {orig_idx+1:02d} -> {src_line}")
                    found = True
                    
            if not found:
                print("    (No dependency needed, 0=0 holds automatically)")

print(f"\n Execution complete! Total time: {time.time() - start_time:.2f} seconds")

Starting to reveal the exact linear superposition relationship between constraints...
Constraint space established...
Solving general solution space for Program 1...

 ============ Algebraic Perspective Results ==============

 Target Program 2, Line 1: H1[1,1] == H0[0,0]/3 - H0[1,1]/3 + H1[0,0]

  For the disassembled [Real Part] constraint:
     Coefficient [ -1/3  ] * Prog 1 Line 13 -> cp.real(H0[0,0]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2
     Coefficient [  1/3  ] * Prog 1 Line 14 -> cp.real(H0[1,1]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2
     Coefficient [  -1   ] * Prog 1 Line 44 -> cp.real(H1[0,0]) == cp.real(H1[4,4])
     Coefficient [   1   ] * Prog 1 Line 45 -> cp.real(H1[1,1]) == cp.real(H1[3,3])

 Target Program 